# 대청댐 유해남조류 발생 예측 AI 모델

## 개요
- **목표**: 조류경보제 발령기준(유해남조류 4종) 발생 및 경보 발령 시점 **7일 선행 예측**
- **대상 지점**: 대청댐 (문의, 추동, 회남 3개 채수위치)
- **예측 대상**: 발령단계 (미발령 / 관심 / 경계 / 조류대발생)
- **데이터 기간**: 2016-01-05 ~ 2025-12-22

## 발령 기준 (기후에너지환경부 조류경보제 운영지침)
| 단계 | 유해남조류 세포수 (cells/mL) | 적용 조건 |
|------|---------------------------|----------|
| 관심 | **1,000 이상** | 2회 연속 기준 초과 시 발령 |
| 경계 | **10,000 이상** | 2회 연속 기준 초과 시 발령 |
| 조류대발생 | **1,000,000 이상** | 2회 연속 기준 초과 시 발령 |
| 해제 | 기준 미만 | 2회 연속 기준 미만 시 해제 |

※ 유해남조류 4종: Microcystis, Anabaena, Oscillatoria, Aphanizomenon

## 데이터 출처
1. **조류 모니터링 데이터**: 한국수자원공사 (K-water) 대청댐 주간 조류 모니터링
2. **기상 데이터**: 기상청 기상자료개방포털 (대전·청주·보은 관측소)
3. **수문/댐운영 데이터**: 한국수자원공사 대청댐 운영 수문정보

## 0. 환경 설정 및 라이브러리 임포트

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib import font_manager, rc
import platform

# 한글 폰트 설정
if platform.system() == 'Windows':
    font_path = 'C:/Windows/Fonts/malgun.ttf'
    font_name = font_manager.FontProperties(fname=font_path).get_name()
    rc('font', family=font_name)
elif platform.system() == 'Darwin':
    rc('font', family='AppleGothic')
else:
    rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, recall_score, precision_score
)
from sklearn.utils.class_weight import compute_sample_weight
import lightgbm as lgb
import xgboost as xgb
import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE = 42
LEAD_DAYS    = 7       # 선행 예측 일수
CONSEC_DAYS  = 7       # 연속 측정 주기 (주 1회 → 7일)

# 발령 임계값 (cells/mL)
THRESHOLDS = {
    '관심':     1_000,
    '경계':     10_000,
    '조류대발생': 1_000_000,
}

ALERT_ORDER = ['미발령', '관심', '경계', '조류대발생']
ALERT2IDX   = {k: i for i, k in enumerate(ALERT_ORDER)}
IDX2ALERT   = {i: k for k, i in ALERT2IDX.items()}

print('라이브러리 로드 완료')
print(f'예측 선행 기간: {LEAD_DAYS}일 / 연속 기준 주기: {CONSEC_DAYS}일')

## 1. 데이터 로딩 및 기본 전처리

In [ ]:
# 컬럼명 위치 기반 매핑 (utf-8-sig CSV 한글 깨짐 대응)
COL_NAMES = [
    '조사일', '채수위치', 'total_cyano', 'microcystis', 'anabaena',
    'oscillatoria', 'aphanizomenon', '일조시간합계(hr)', '투명도', '발령단계',
    '수온(℃)', 'pH', 'DO(㎎/L)', '탁도', 'Chl-a(㎎/㎥)',
    '평균기온(°C)', '최저기온(°C)', '최고기온(°C)', '평균풍속(m/s)',
    '평균상대습도(%)', '합계일조시간(hr)', '합계일사량(MJ/m2)', '평균전운량(1/10)',
    '저수위(EL.m)', '저수량(백만㎥)', '저수율(%)', '강우량(mm)',
    '유입량(㎥/s)', '총방류량(㎥/s)', '일강수량(mm)'
]

df_raw = pd.read_csv('finaldata.csv', encoding='utf-8-sig', header=0)
df_raw.columns = COL_NAMES
df_raw['조사일'] = pd.to_datetime(df_raw['조사일'])
df_raw = df_raw.sort_values(['채수위치', '조사일']).reset_index(drop=True)

# 채수위치 한글 복원 (순서: 문의, 추동, 회남)
unique_sites = df_raw['채수위치'].unique()
if len(unique_sites) == 3:
    site_rename = dict(zip(unique_sites, ['문의', '추동', '회남']))
    df_raw['채수위치'] = df_raw['채수위치'].map(site_rename)

# ── 2회 연속 기준 적용한 발령단계 재계산 ──────────────────────────────
# 조류경보제 운영지침: 주 1회 측정, 2회 연속 임계값 초과 시 경보 발령
# 일별 확장 데이터에서: 현재값과 7일 전 값이 모두 임계값 이상이면 발령
def add_consecutive_alert(df, consec_days=7):
    df = df.copy().sort_values(['채수위치', '조사일'])
    levels = []
    for site in df['채수위치'].unique():
        mask = df['채수위치'] == site
        s    = df.loc[mask, 'total_cyano'].values
        prev = np.concatenate([[np.nan] * consec_days, s[:-consec_days]])
        for cur, pre in zip(s, prev):
            cur_v  = float(cur)  if not pd.isna(cur)  else 0.0
            prev_v = float(pre)  if not pd.isna(pre)  else 0.0
            pair_min = min(cur_v, prev_v)
            if pair_min >= THRESHOLDS['조류대발생']:
                levels.append('조류대발생')
            elif pair_min >= THRESHOLDS['경계']:
                levels.append('경계')
            elif pair_min >= THRESHOLDS['관심']:
                levels.append('관심')
            else:
                levels.append('미발령')
    df['발령단계_2회'] = levels
    return df

df_raw = add_consecutive_alert(df_raw, consec_days=CONSEC_DAYS)

print(f'데이터 크기: {df_raw.shape}')
print(f'기간: {df_raw["조사일"].min().date()} ~ {df_raw["조사일"].max().date()}')
print(f'채수위치: {df_raw["채수위치"].unique().tolist()}')
print(f'\n발령단계 분포 (2회 연속 기준):')
print(df_raw['발령단계_2회'].value_counts())
df_raw.head(3)

## 2. 탐색적 데이터 분석 (EDA)

In [ ]:
df_raw['연도'] = df_raw['조사일'].dt.year
df_raw['월']  = df_raw['조사일'].dt.month

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1) 시계열 total_cyano
ax = axes[0, 0]
for site in ['문의', '추동', '회남']:
    sub = df_raw[df_raw['채수위치'] == site]
    ax.plot(sub['조사일'], sub['total_cyano'], label=site, alpha=0.7, linewidth=0.8)
ax.axhline(THRESHOLDS['관심'],      color='orange', linestyle='--', linewidth=1.2, label='관심 기준(1,000)')
ax.axhline(THRESHOLDS['경계'],      color='red',    linestyle='--', linewidth=1.2, label='경계 기준(10,000)')
ax.axhline(THRESHOLDS['조류대발생'], color='purple', linestyle='--', linewidth=1.2, label='대발생 기준(1,000,000)')
ax.set_yscale('log')
ax.set_title('유해남조류 세포수 시계열 (log scale)', fontsize=13)
ax.set_xlabel('날짜'); ax.set_ylabel('세포수 (cells/mL)')
ax.legend(fontsize=8)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# 2) 월별 평균 세포수
ax = axes[0, 1]
monthly = df_raw.groupby(['월', '채수위치'])['total_cyano'].mean().reset_index()
for site in ['문의', '추동', '회남']:
    s = monthly[monthly['채수위치'] == site]
    ax.plot(s['월'], s['total_cyano'], marker='o', label=site)
ax.set_title('월별 평균 유해남조류 세포수', fontsize=13)
ax.set_xlabel('월'); ax.set_ylabel('평균 세포수 (cells/mL)')
ax.set_xticks(range(1, 13))
ax.legend()

# 3) 연도별 발령단계 비율 (2회 연속 기준)
ax = axes[1, 0]
pivot = df_raw.groupby(['연도', '발령단계_2회']).size().unstack(fill_value=0)
for col in ALERT_ORDER:
    if col not in pivot.columns:
        pivot[col] = 0
pivot = pivot[ALERT_ORDER]
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
colors = ['#4CAF50', '#FFC107', '#FF5722', '#9C27B0']
pivot_pct.plot(kind='bar', stacked=True, ax=ax, color=colors)
ax.set_title('연도별 발령단계 비율 (2회 연속 기준)', fontsize=13)
ax.set_xlabel('연도'); ax.set_ylabel('비율 (%)')
ax.tick_params(axis='x', rotation=45)
ax.legend(loc='upper left', fontsize=8)

# 4) 4종 조류 구성비
ax = axes[1, 1]
species = ['microcystis', 'anabaena', 'oscillatoria', 'aphanizomenon']
ssum = df_raw[species].sum()
ax.pie(ssum, labels=['Microcystis', 'Anabaena', 'Oscillatoria', 'Aphanizomenon'],
       autopct='%1.1f%%', colors=['#e74c3c','#3498db','#2ecc71','#f39c12'],
       startangle=90, textprops={'fontsize': 10})
ax.set_title('유해남조류 4종 구성비 (전체 기간)', fontsize=13)

plt.suptitle('대청댐 유해남조류 탐색적 데이터 분석 (EDA)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA 그래프 저장: eda_overview.png')

In [ ]:
# 주요 변수 상관관계
corr_cols = [
    'total_cyano', '수온(℃)', 'pH', 'DO(㎎/L)', '탁도', 'Chl-a(㎎/㎥)',
    '평균기온(°C)', '최고기온(°C)', '합계일사량(MJ/m2)', '합계일조시간(hr)',
    '평균상대습도(%)', '일강수량(mm)', '저수위(EL.m)', '저수율(%)',
    '유입량(㎥/s)', '총방류량(㎥/s)', '평균전운량(1/10)'
]
corr_matrix = df_raw[corr_cols].dropna().corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, annot_kws={'size': 8}, linewidths=0.5)
ax.set_title('주요 변수 피어슨 상관계수 행렬', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('total_cyano와의 상관관계 (절댓값 기준 상위 10):')
print(corr_matrix['total_cyano'].abs().sort_values(ascending=False)[1:11])

## 3. 피처 엔지니어링

- **Lag 피처**: 1, 3, 7, 14일 전 값 (과거 조류 상태 반영)
- **Rolling 통계**: 7, 14, 30일 이동 평균·최대·표준편차
- **계절 주기 인코딩**: sin/cos 변환으로 월 계절성 반영
- **교호작용**: 수온 × 일사량 (조류 성장 최적 환경)
- **선행 타겟**: N일 후 total_cyano 및 발령단계

In [ ]:
def build_features(df, lead_days=7, consec_days=7):
    df = df.copy().sort_values(['채수위치', '조사일'])

    # 1. 시간 피처
    df['year']       = df['조사일'].dt.year
    df['month']      = df['조사일'].dt.month
    df['dayofyear']  = df['조사일'].dt.dayofyear
    df['weekofyear'] = df['조사일'].dt.isocalendar().week.astype(int)
    df['month_sin']  = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']  = np.cos(2 * np.pi * df['month'] / 12)
    df['doy_sin']    = np.sin(2 * np.pi * df['dayofyear'] / 365)
    df['doy_cos']    = np.cos(2 * np.pi * df['dayofyear'] / 365)
    df['is_summer']  = df['month'].between(6, 9).astype(int)

    # 2. Lag 피처
    lag_cols = ['total_cyano', '수온(℃)', '평균기온(°C)', '합계일사량(MJ/m2)',
                '일강수량(mm)', 'Chl-a(㎎/㎥)', 'pH', 'DO(㎎/L)', '탁도', '저수위(EL.m)', '유입량(㎥/s)']
    for col in lag_cols:
        if col in df.columns:
            for lag in [1, 3, 7, 14]:
                df[f'{col}_lag{lag}'] = df.groupby('채수위치')[col].shift(lag)

    # 3. Rolling 통계
    roll_cols = ['total_cyano', '수온(℃)', '평균기온(°C)', '합계일사량(MJ/m2)', '일강수량(mm)', 'Chl-a(㎎/㎥)']
    for col in roll_cols:
        if col in df.columns:
            for w in [7, 14, 30]:
                grp = df.groupby('채수위치')[col]
                mn = max(1, w // 2)
                df[f'{col}_r{w}_mean'] = grp.transform(lambda x: x.shift(1).rolling(w, min_periods=mn).mean())
                df[f'{col}_r{w}_max']  = grp.transform(lambda x: x.shift(1).rolling(w, min_periods=mn).max())
                df[f'{col}_r{w}_std']  = grp.transform(lambda x: x.shift(1).rolling(w, min_periods=mn).std())

    # 4. 누적 강수량
    if '일강수량(mm)' in df.columns:
        for w in [3, 7, 14]:
            df[f'cum_rain_{w}d'] = df.groupby('채수위치')['일강수량(mm)'].transform(
                lambda x: x.shift(1).rolling(w, min_periods=1).sum())

    # 5. 교호작용: 수온 × 일사량
    if '수온(℃)' in df.columns and '합계일사량(MJ/m2)' in df.columns:
        df['temp_x_solar'] = df['수온(℃)'] * df['합계일사량(MJ/m2)']

    # 6. 방류/유입 비율
    if '총방류량(㎥/s)' in df.columns and '유입량(㎥/s)' in df.columns:
        df['outflow_inflow_ratio'] = df['총방류량(㎥/s)'] / (df['유입량(㎥/s)'] + 1e-6)

    # 7. log total_cyano (현재, 피처로 사용)
    df['log_cyano_now'] = np.log1p(df['total_cyano'])

    # 8. 이진 경보 (현재)
    df['alert_binary_now'] = (df['total_cyano'] >= THRESHOLDS['관심']).astype(int)

    # 9. 2회 연속 기준 발령단계 현재값 (카테고리 인코딩)
    df['alert_level_now'] = df['발령단계_2회'].map(ALERT2IDX).fillna(0).astype(int)

    # ── 선행 타겟 생성 (N일 후) ──
    # total_cyano 값 (회귀 보조)
    df[f'target_cyano_{lead_days}d'] = df.groupby('채수위치')['total_cyano'].transform(
        lambda x: x.shift(-lead_days))

    # 발령단계 (2회 연속 기준) → N일 후
    df[f'target_alert_{lead_days}d'] = df.groupby('채수위치')['발령단계_2회'].transform(
        lambda x: x.shift(-lead_days))

    # 이진 (경보 발령 여부)
    df[f'target_binary_{lead_days}d'] = df.groupby('채수위치')['alert_binary_now'].transform(
        lambda x: x.shift(-lead_days))

    return df


df_feat = build_features(df_raw, lead_days=LEAD_DAYS, consec_days=CONSEC_DAYS)
print(f'피처 엔지니어링 완료: {df_feat.shape[0]:,}행 × {df_feat.shape[1]}열')
print(f'추가 피처 수: {df_feat.shape[1] - df_raw.shape[1]}')

In [ ]:
TARGET_COL = f'target_alert_{LEAD_DAYS}d'
TARGET_BIN = f'target_binary_{LEAD_DAYS}d'
TARGET_REG = f'target_cyano_{LEAD_DAYS}d'

# 피처 컬럼 선정 (미래 정보 및 타겟 컬럼 제외)
drop_cols = [
    '조사일', '채수위치', '발령단계', '발령단계_2회',
    'total_cyano', 'microcystis', 'anabaena', 'oscillatoria', 'aphanizomenon',
    '연도', '월', '일조시간합계(hr)',
    TARGET_COL, TARGET_BIN, TARGET_REG
]
feature_cols = [c for c in df_feat.columns if c not in drop_cols]

# 타겟 결측 제거
df_model = df_feat.dropna(subset=[TARGET_COL]).copy()
df_model['target_encoded'] = df_model[TARGET_COL].map(ALERT2IDX)
df_model = df_model.dropna(subset=['target_encoded'])
df_model['target_encoded'] = df_model['target_encoded'].astype(int)

X = df_model[feature_cols].copy()
y = df_model['target_encoded'].copy()

# 실제 데이터에 존재하는 클래스 확인
present_classes = sorted(y.unique())
N_CLASSES = 4  # 항상 4클래스 모델 유지 (조류대발생 포함)
present_labels = [IDX2ALERT[i] for i in range(N_CLASSES)]

print(f'학습용 데이터: {X.shape[0]:,}행 × {X.shape[1]}피처')
print(f'\n타겟 분포 (2회 연속 기준):')
for i, label in enumerate(ALERT_ORDER):
    cnt = (y == i).sum()
    print(f'  [{i}] {label:8s}: {cnt:5,}  ({cnt/len(y)*100:.1f}%)')

print(f'\n※ 조류대발생(>=1,000,000 cells/mL) 클래스는 역사적 데이터에서 미관측')
print(f'   → 모델은 num_class=4로 설정, 예측 시 threshold 후처리로 보완')

## 4. 시계열 분할 (Train / Validation / Test)

In [ ]:
# 시간 순서 유지 분할: Train 2016-2021 / Val 2022-2023 / Test 2024-2025
df_ms = df_model.sort_values('조사일')
X_s   = df_ms[feature_cols].copy()
y_s   = df_ms['target_encoded'].copy()
dates = df_ms['조사일'].copy()

tr_mask  = dates.dt.year <= 2021
val_mask = dates.dt.year.between(2022, 2023)
te_mask  = dates.dt.year >= 2024

X_train, y_train = X_s[tr_mask],  y_s[tr_mask]
X_val,   y_val   = X_s[val_mask], y_s[val_mask]
X_test,  y_test  = X_s[te_mask],  y_s[te_mask]

print(f'Train: {len(X_train):,} ({dates[tr_mask].min().date()} ~ {dates[tr_mask].max().date()})')
print(f'Val  : {len(X_val):,} ({dates[val_mask].min().date()} ~ {dates[val_mask].max().date()})')
print(f'Test : {len(X_test):,} ({dates[te_mask].min().date()} ~ {dates[te_mask].max().date()})')

print('\n훈련 세트 클래스 분포:')
for i, label in enumerate(ALERT_ORDER):
    cnt = (y_train == i).sum()
    print(f'  {label:8s}: {cnt:,}')

## 5. LightGBM 모델 학습 (주 모델)

- `num_class=4` 고정 (조류대발생 클래스 포함)
- `class_weight='balanced'`로 클래스 불균형 보정
- Optuna로 50회 하이퍼파라미터 탐색

In [ ]:
def lgb_objective(trial):
    params = {
        'objective':         'multiclass',
        'num_class':         N_CLASSES,
        'metric':            'multi_logloss',
        'verbosity':         -1,
        'random_state':      RANDOM_STATE,
        'n_estimators':      trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth':         trial.suggest_int('max_depth', 4, 10),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'class_weight':      'balanced',
    }
    m = lgb.LGBMClassifier(**params)
    m.fit(X_train, y_train,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    preds = m.predict(X_val)
    return f1_score(y_val, preds, average='macro', zero_division=0)

study = optuna.create_study(direction='maximize', study_name='lgb_algae')
study.optimize(lgb_objective, n_trials=50, show_progress_bar=True)

print(f'\n최적 Macro F1 (Validation): {study.best_value:.4f}')
print(f'최적 파라미터: {study.best_params}')

In [ ]:
best_params = study.best_params.copy()
best_params.update({
    'objective':    'multiclass',
    'num_class':    N_CLASSES,
    'metric':       'multi_logloss',
    'verbosity':    -1,
    'random_state': RANDOM_STATE,
    'class_weight': 'balanced',
})

lgb_model = lgb.LGBMClassifier(**best_params)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)]
)

y_pred_lgb = lgb_model.predict(X_test)
y_prob_lgb = lgb_model.predict_proba(X_test)  # shape: (n_samples, N_CLASSES)

# predict_proba 열 수가 모델 내부 클래스 수와 다를 수 있으므로 4열로 통일
if y_prob_lgb.shape[1] < 4:
    pad = np.zeros((len(y_prob_lgb), 4 - y_prob_lgb.shape[1]))
    y_prob_lgb = np.hstack([y_prob_lgb, pad])

print('=== LightGBM 테스트 성능 (7일 선행 예측) ===')
print(classification_report(
    y_test, y_pred_lgb,
    labels=list(range(N_CLASSES)),
    target_names=ALERT_ORDER,
    zero_division=0
))

In [ ]:
sw = compute_sample_weight('balanced', y_train)

xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=N_CLASSES,
    n_estimators=500, learning_rate=0.05,
    max_depth=6, subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, eval_metric='mlogloss',
    verbosity=0, early_stopping_rounds=50
)
xgb_model.fit(
    X_train, y_train,
    sample_weight=sw,
    eval_set=[(X_val, y_val)],
    verbose=False
)
y_pred_xgb = xgb_model.predict(X_test)

print('=== XGBoost 테스트 성능 (7일 선행 예측) ===')
print(classification_report(
    y_test, y_pred_xgb,
    labels=list(range(N_CLASSES)),
    target_names=ALERT_ORDER,
    zero_division=0
))

## 6. 모델 평가 및 성능 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, preds, title in zip(axes, [y_pred_lgb, y_pred_xgb], ['LightGBM', 'XGBoost']):
    cm = confusion_matrix(y_test, preds, labels=list(range(N_CLASSES)))
    rs = cm.sum(axis=1, keepdims=True)
    rs[rs == 0] = 1
    cm_pct = cm.astype(float) / rs * 100
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=ALERT_ORDER, yticklabels=ALERT_ORDER, ax=ax)
    ax.set_title(f'{title} 혼동행렬 (%, {LEAD_DAYS}일 선행)', fontsize=12)
    ax.set_xlabel('예측 발령단계'); ax.set_ylabel('실제 발령단계')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

metrics = []
for name, preds in [('LightGBM', y_pred_lgb), ('XGBoost', y_pred_xgb)]:
    metrics.append({
        '모델': name,
        'Accuracy':   f'{accuracy_score(y_test, preds):.4f}',
        'Macro F1':   f'{f1_score(y_test, preds, average="macro", zero_division=0):.4f}',
        'Weighted F1':f'{f1_score(y_test, preds, average="weighted", zero_division=0):.4f}',
        '경계 Recall':f'{recall_score(y_test, preds, labels=[2], average="macro", zero_division=0):.4f}',
    })
print('\n=== 모델 성능 비교 ===')
print(pd.DataFrame(metrics).to_string(index=False))

In [ ]:
# 리드타임별 성능 분석 (1, 3, 5, 7, 14일)
lead_results = []

for ld in [1, 3, 5, 7, 14]:
    df_ld = build_features(df_raw, lead_days=ld, consec_days=CONSEC_DAYS)
    tc = f'target_alert_{ld}d'
    dm = df_ld.dropna(subset=[tc]).copy()
    dm['te'] = dm[tc].map(ALERT2IDX)
    dm = dm.dropna(subset=['te'])
    dm['te'] = dm['te'].astype(int)

    fc_ld = [c for c in feature_cols if c in dm.columns]
    tr = dm[dm['조사일'].dt.year <= 2021]
    va = dm[dm['조사일'].dt.year.between(2022, 2023)]
    te = dm[dm['조사일'].dt.year >= 2024]
    if len(tr) == 0 or len(te) == 0:
        continue

    p = best_params.copy()
    p['num_class'] = N_CLASSES
    m = lgb.LGBMClassifier(**p)
    m.fit(tr[fc_ld], tr['te'], eval_set=[(va[fc_ld], va['te'])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    preds = m.predict(te[fc_ld])

    lead_results.append({
        '리드타임(일)': ld,
        'Accuracy':  accuracy_score(te['te'], preds),
        'Macro F1':  f1_score(te['te'], preds, average='macro', zero_division=0),
        '경계_Recall': recall_score(te['te'], preds, labels=[2], average='macro', zero_division=0),
    })
    print(f'  리드타임 {ld:2d}일 → Macro F1: {lead_results[-1]["Macro F1"]:.4f}')

lead_df = pd.DataFrame(lead_results)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(lead_df['리드타임(일)'], lead_df['Macro F1'], 'o-', label='Macro F1', lw=2)
ax.plot(lead_df['리드타임(일)'], lead_df['Accuracy'], 's--', label='Accuracy', lw=2)
ax.plot(lead_df['리드타임(일)'], lead_df['경계_Recall'], '^:', label='경계 Recall', lw=2)
ax.set_title('리드타임에 따른 모델 성능 변화 (LightGBM)', fontsize=13)
ax.set_xlabel('선행 예측 일수 (일)'); ax.set_ylabel('성능 지표')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lead_time_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n리드타임별 성능:')
print(lead_df.to_string(index=False))

In [ ]:
# 경보 적중률 분석
test_dates_r = dates[te_mask].reset_index(drop=True)
y_test_r     = y_test.reset_index(drop=True)
y_pred_r     = pd.Series(y_pred_lgb)

res = pd.DataFrame({
    '조사일':  test_dates_r.values,
    '실제코드': y_test_r.values,
    '예측코드': y_pred_lgb,
    '실제':    y_test_r.map(IDX2ALERT),
    '예측':    pd.Series(y_pred_lgb).map(IDX2ALERT),
})

actual_alert  = res[res['실제코드'] >= 1]
actual_normal = res[res['실제코드'] == 0]
hit       = (actual_alert['예측코드'] >= 1).sum()
miss      = len(actual_alert) - hit
false_alm = (actual_normal['예측코드'] >= 1).sum()

print(f'=== {LEAD_DAYS}일 선행 예측 경보 적중률 (LightGBM) ===')
print(f'테스트 기간: {res["조사일"].min().date()} ~ {res["조사일"].max().date()}')
print(f'실제 경보 발령 일수: {len(actual_alert):,}일')
print(f'경보 적중 (Hit):     {hit:,}일 ({hit/max(len(actual_alert),1)*100:.1f}%)')
print(f'경보 누락 (Miss):    {miss:,}일 ({miss/max(len(actual_alert),1)*100:.1f}%)')
print(f'오경보 (False Alarm):{false_alm:,}일 ({false_alm/max(len(actual_normal),1)*100:.1f}%)')
print('\n등급별 적중률:')
for code, label in {1:'관심', 2:'경계', 3:'조류대발생'}.items():
    sub = res[res['실제코드'] == code]
    if len(sub) > 0:
        h = (sub['예측코드'] >= 1).sum()
        print(f'  {label:8s}: {h}/{len(sub)} ({h/len(sub)*100:.1f}%)')

## 7. 임계값 기반 조류대발생 후처리

역사적 데이터에 `조류대발생` 샘플이 없으므로,  
회귀 모델로 예측한 `total_cyano ≥ 1,000,000`이면 `조류대발생`으로 상향 조정합니다.

In [ ]:
# ── total_cyano 회귀 모델 (log 스케일) ──
y_train_reg = np.log1p(df_ms[tr_mask][TARGET_REG].fillna(0))
y_val_reg   = np.log1p(df_ms[val_mask][TARGET_REG].fillna(0))
y_test_reg  = np.log1p(df_ms[te_mask][TARGET_REG].fillna(0))

reg_params = best_params.copy()
reg_params.update({'objective': 'regression_l1', 'metric': 'mae'})
reg_params.pop('num_class', None)
reg_params.pop('class_weight', None)

reg_model = lgb.LGBMRegressor(**reg_params)
reg_model.fit(
    X_train, y_train_reg,
    eval_set=[(X_val, y_val_reg)],
    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)]
)

y_pred_cyano_log = reg_model.predict(X_test)
y_pred_cyano     = np.expm1(y_pred_cyano_log)

print('회귀 모델 예측값 통계 (cells/mL):')
print(f'  min={y_pred_cyano.min():.1f}, max={y_pred_cyano.max():.1f}, mean={y_pred_cyano.mean():.1f}')
print(f'  조류대발생 범위(>=1,000,000) 예측 비율: {(y_pred_cyano >= 1e6).mean()*100:.2f}%')

In [ ]:
# ── 조합 예측: 분류 결과 + 회귀 threshold 후처리 ──
def apply_threshold_postprocess(y_pred_cls, y_pred_cyano_val, consec_days=7):
    """
    분류 예측에 threshold 기반 후처리 적용:
    - 회귀 예측값이 조류대발생 임계값 이상이면 class=3으로 상향
    - 경계/관심 임계값 기반 교차 확인
    """
    y_final = y_pred_cls.copy()

    # 조류대발생 조건 (회귀 예측 >= 1,000,000)
    mask_major = y_pred_cyano_val >= THRESHOLDS['조류대발생']
    y_final[mask_major] = 3

    # 경계 미달인데 경계 이상 예측된 경우: 회귀로 cross-check
    mask_경계_check = (y_pred_cls >= 2) & (y_pred_cyano_val < THRESHOLDS['경계'])
    y_final[mask_경계_check] = np.minimum(y_pred_cls[mask_경계_check], 1)

    return y_final

y_pred_combined = apply_threshold_postprocess(
    y_pred_lgb.copy(), y_pred_cyano
)

print('=== 조합 예측 (분류 + 회귀 후처리) 테스트 성능 ===')
print(classification_report(
    y_test, y_pred_combined,
    labels=list(range(N_CLASSES)),
    target_names=ALERT_ORDER,
    zero_division=0
))

## 8. SHAP 기반 주요 영향 인자 분석

In [ ]:
explainer = shap.TreeExplainer(lgb_model)

sample_n = min(2000, len(X_test))
X_shap = X_test.sample(sample_n, random_state=RANDOM_STATE)

shap_raw = explainer.shap_values(X_shap)

# SHAP 반환 형태 통일
# 구버전: list of (n_samples, n_features) × n_classes
# 신버전: ndarray (n_samples, n_features, n_classes)
if isinstance(shap_raw, list):
    shap_values = shap_raw
    n_shap_cls = len(shap_values)
elif shap_raw.ndim == 3:
    n_shap_cls = shap_raw.shape[2]
    shap_values = [shap_raw[:, :, c] for c in range(n_shap_cls)]
else:
    # 2D: single class (binary) – wrap
    shap_values = [shap_raw]
    n_shap_cls = 1

print(f'SHAP 계산 완료: {X_shap.shape}')
print(f'SHAP 클래스 수: {n_shap_cls}')

In [ ]:
# 전체 클래스 평균 절댓값 SHAP 중요도
shap_abs = np.mean([np.abs(shap_values[c]) for c in range(n_shap_cls)], axis=0)
shap_importance = pd.DataFrame({
    'feature':          X_shap.columns,
    'shap_importance':  shap_abs.mean(axis=0)
}).sort_values('shap_importance', ascending=False).reset_index(drop=True)

top_n = 25
fig, ax = plt.subplots(figsize=(10, 9))
top_f = shap_importance.head(top_n)
colors_bar = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, top_n))
ax.barh(range(top_n), top_f['shap_importance'].values[::-1], color=colors_bar)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_f['feature'].values[::-1], fontsize=10)
ax.set_xlabel('평균 |SHAP 값|', fontsize=11)
ax.set_title(f'유해남조류 발생 예측 주요 영향 인자 Top {top_n}\n(SHAP Feature Importance)', fontsize=13)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n상위 10개 주요 영향 인자:')
print(shap_importance.head(10).to_string(index=False))

In [ ]:
# SHAP Beeswarm - 경계 클래스
cls_idx = min(2, n_shap_cls - 1)  # 경계 클래스 (index 2)
sv_cls = shap_values[cls_idx]

top10 = shap_importance.head(10)['feature'].tolist()
col_idx = [X_shap.columns.tolist().index(c) for c in top10]

exp = shap.Explanation(
    values=sv_cls[:, col_idx],
    base_values=explainer.expected_value[cls_idx] if hasattr(explainer.expected_value, '__len__') else explainer.expected_value,
    data=X_shap[top10].values,
    feature_names=top10
)

plt.figure(figsize=(10, 7))
shap.plots.beeswarm(exp, max_display=10, show=False)
plt.title('SHAP Beeswarm Plot - 경계 단계 예측 영향 인자', fontsize=13)
plt.tight_layout()
plt.savefig('shap_beeswarm_경계.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 클래스별 SHAP 중요도
n_show_cls = min(n_shap_cls, 3)
cls_show   = list(range(1, n_show_cls + 1))   # 관심, 경계, (조류대발생)
cls_labels_show = [ALERT_ORDER[i] for i in cls_show if i < len(ALERT_ORDER)]

fig, axes = plt.subplots(1, len(cls_labels_show), figsize=(6 * len(cls_labels_show), 6))
if len(cls_labels_show) == 1:
    axes = [axes]

cls_colors = {'관심': '#f1c40f', '경계': '#e67e22', '조류대발생': '#e74c3c'}
for ax, ci, cl in zip(axes, cls_show, cls_labels_show):
    if ci >= n_shap_cls:
        ax.text(0.5, 0.5, f'{cl}\n데이터 없음', ha='center', va='center', transform=ax.transAxes)
        continue
    sv = shap_values[ci]
    imp = np.abs(sv).mean(axis=0)
    top_idx = np.argsort(imp)[::-1][:10]
    ax.barh(range(10), imp[top_idx][::-1], color=cls_colors.get(cl, '#3498db'))
    ax.set_yticks(range(10))
    ax.set_yticklabels([X_shap.columns[i] for i in top_idx][::-1], fontsize=9)
    ax.set_title(f'[{cl}] 영향 인자 Top 10', fontsize=11)
    ax.set_xlabel('평균 |SHAP|', fontsize=9)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('발령단계별 주요 영향 인자 (SHAP)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. 예측 시각화 및 경보 발령 타임라인

In [ ]:
test_df = df_ms[te_mask].copy().reset_index(drop=True)
test_df['예측단계']    = pd.Series(y_pred_combined).map(IDX2ALERT)
test_df['실제단계']    = y_test.reset_index(drop=True).map(IDX2ALERT)
test_df['예측_cyano'] = y_pred_cyano

site_test = test_df[test_df['채수위치'] == '문의'].copy()

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
color_map  = {'미발령': '#27ae60', '관심': '#f39c12', '경계': '#e74c3c', '조류대발생': '#8e44ad'}
code_map   = {'미발령': 0, '관심': 1, '경계': 2, '조류대발생': 3}

# (1) 실제 세포수
ax = axes[0]
ax.plot(site_test['조사일'], site_test['total_cyano'], color='steelblue', lw=1, label='실제 세포수')
ax.plot(site_test['조사일'], site_test['예측_cyano'],  color='coral',     lw=1, alpha=0.7, linestyle='--', label='예측 세포수')
for thresh, clr, lbl in [
    (THRESHOLDS['관심'],     'orange', '관심'),
    (THRESHOLDS['경계'],     'red',    '경계'),
    (THRESHOLDS['조류대발생'], 'purple', '대발생'),
]:
    ax.axhline(thresh, color=clr, linestyle='--', lw=1, label=f'{lbl} 기준({thresh:,})')
ax.set_yscale('log'); ax.set_ylabel('세포수 (log scale)', fontsize=10)
ax.set_title('대청댐(문의) 테스트 기간 예측 결과 (2024~2025)', fontsize=13)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# (2) 실제 발령단계
ax = axes[1]
rc = [code_map.get(v, 0) for v in site_test['실제단계']]
ax.scatter(site_test['조사일'], rc, c=[color_map.get(v, '#27ae60') for v in site_test['실제단계']], s=15, alpha=0.8)
ax.set_yticks([0,1,2,3]); ax.set_yticklabels(ALERT_ORDER, fontsize=9)
ax.set_ylabel('실제 단계', fontsize=10); ax.grid(True, alpha=0.3)

# (3) 예측 발령단계
ax = axes[2]
pc = [code_map.get(v, 0) for v in site_test['예측단계']]
ax.scatter(site_test['조사일'], pc, c=[color_map.get(v, '#27ae60') for v in site_test['예측단계']], s=15, alpha=0.8, marker='s')
ax.set_yticks([0,1,2,3]); ax.set_yticklabels(ALERT_ORDER, fontsize=9)
ax.set_ylabel(f'{LEAD_DAYS}일 선행 예측', fontsize=10)
ax.set_xlabel('날짜', fontsize=10); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('prediction_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. 시나리오 기반 의사결정 지원 시스템 (DSS)

사전 시나리오를 설정하여 기상·수질·운영 조건 변화에 따른  
7일 후 조류경보 발령 확률을 즉시 산출합니다.

In [ ]:
def predict_scenario(base_date, scenario_params, model=lgb_model, reg=reg_model):
    """
    시나리오 조건 변경 후 7일 후 경보 발령 확률 예측.
    Parameters
    ----------
    base_date : str  예측 기준일
    scenario_params : dict  변경할 피처 이름과 값
    """
    base_date = pd.to_datetime(base_date)
    ref = df_feat[df_feat['조사일'] <= base_date].sort_values('조사일').tail(1)
    if len(ref) == 0:
        raise ValueError('기준일 이전 데이터 없음')

    X_sc = ref[feature_cols].copy()
    for col, val in scenario_params.items():
        for fc in X_sc.columns:
            if col in fc:
                X_sc[fc] = val

    proba = model.predict_proba(X_sc)[0]
    if len(proba) < 4:
        proba = np.append(proba, [0.0] * (4 - len(proba)))

    pred_cyano = float(np.expm1(reg.predict(X_sc)[0]))
    pred_cls   = int(np.argmax(proba))

    # threshold 후처리
    if pred_cyano >= THRESHOLDS['조류대발생']:
        pred_cls = 3

    return {
        '기준일':   base_date.date(),
        '예측일':   (base_date + pd.Timedelta(days=LEAD_DAYS)).date(),
        '예측단계': ALERT_ORDER[pred_cls],
        '예측_세포수(cells/mL)': f'{pred_cyano:,.0f}',
        '미발령%': f'{proba[0]*100:.1f}',
        '관심%':   f'{proba[1]*100:.1f}',
        '경계%':   f'{proba[2]*100:.1f}',
        '대발생%': f'{proba[3]*100:.1f}',
    }, proba


BASE_DATE = '2024-07-15'
SCENARIOS = {
    '현황 (기준)': {},
    '시나리오A: 폭염 (기온+5°C, 무강수)': {
        '평균기온(°C)': 33, '최고기온(°C)': 38, '일강수량(mm)': 0
    },
    '시나리오B: 집중호우 (150mm 강수)': {
        '일강수량(mm)': 150, '강우량(mm)': 150, '평균상대습도(%)': 95
    },
    '시나리오C: 조류 위험 (고온+저수위+저강수)': {
        '평균기온(°C)': 32, '합계일사량(MJ/m2)': 26,
        '일강수량(mm)': 0, '저수위(EL.m)': 70, '유입량(㎥/s)': 8
    },
    '시나리오D: 방류 증대 (조류 억제)': {
        '총방류량(㎥/s)': 250, '유입량(㎥/s)': 200
    },
}

print(f'=== 시나리오 기반 DSS (기준일: {BASE_DATE}, 예측: +{LEAD_DAYS}일) ===')
sc_probas, sc_names = [], []

for name, params in SCENARIOS.items():
    try:
        r, proba = predict_scenario(BASE_DATE, params)
        sc_probas.append(proba)
        sc_names.append(name.split(':')[0].strip())
        print(f'\n[{name}]')
        print(f'  예측 단계: {r["예측단계"]}  예측 세포수: {r["예측_세포수(cells/mL)"]} cells/mL')
        print(f'  확률 → 미발령:{r["미발령%"]}% | 관심:{r["관심%"]}% | 경계:{r["경계%"]}% | 대발생:{r["대발생%"]}%')
    except Exception as e:
        print(f'  [{name}] 오류: {e}')

In [ ]:
if sc_probas:
    fig, ax = plt.subplots(figsize=(13, 6))
    x = np.arange(len(sc_names))
    w = 0.2
    c_list = ['#27ae60', '#f39c12', '#e74c3c', '#8e44ad']
    for i, (lbl, col) in enumerate(zip(ALERT_ORDER, c_list)):
        ax.bar(x + i * w, [p[i] * 100 for p in sc_probas], w, label=lbl, color=col, alpha=0.85)
    ax.set_xticks(x + w * 1.5)
    ax.set_xticklabels(sc_names, fontsize=10)
    ax.set_ylabel('예측 확률 (%)')
    ax.set_title(f'시나리오별 조류경보 발령 예측 확률\n(기준일: {BASE_DATE}, {LEAD_DAYS}일 선행)', fontsize=13)
    ax.legend(loc='upper right'); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, 105)
    plt.tight_layout()
    plt.savefig('scenario_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 월별 조류경보 위험도 달력
td = df_ms[te_mask].copy().reset_index(drop=True)
td['관심이상_확률'] = y_prob_lgb[:, 1] + y_prob_lgb[:, 2] + y_prob_lgb[:, 3]
td['경계이상_확률'] = y_prob_lgb[:, 2] + y_prob_lgb[:, 3]

mr = td.groupby(td['조사일'].dt.month).agg(
    관심이상=('관심이상_확률', 'mean'),
    경계이상=('경계이상_확률', 'mean')
).reset_index()
mr.columns = ['월', '관심이상', '경계이상']

fig, ax = plt.subplots(figsize=(11, 5))
months = mr['월'].values
ml = ['1월','2월','3월','4월','5월','6월','7월','8월','9월','10월','11월','12월']
ax.bar(months - 0.2, mr['관심이상'] * 100, 0.4, label='관심 이상', color='#f39c12', alpha=0.8)
ax.bar(months + 0.2, mr['경계이상'] * 100, 0.4, label='경계 이상', color='#e74c3c', alpha=0.8)
ax.set_xticks(months)
ax.set_xticklabels([ml[m-1] for m in months], fontsize=10)
ax.set_ylabel('평균 예측 확률 (%)')
ax.set_title('월별 조류경보 위험도 달력 (테스트 기간: 2024~2025)', fontsize=13)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('monthly_risk_calendar.png', dpi=150, bbox_inches='tight')
plt.show()
print('월별 경보 위험도:')
print(mr.to_string(index=False))

## 11. 종합 결과 요약

In [ ]:
print('=' * 70)
print('  대청댐 유해남조류 발생 예측 AI 모델 - 최종 결과 요약')
print('=' * 70)
print()
print('▶ 모델 구성')
print(f'  - 분류 모델: LightGBM (Optuna 50-trial HPO) + XGBoost (비교)')
print(f'  - 회귀 모델: LightGBM (total_cyano log 예측) → threshold 후처리')
print(f'  - 선행 예측: {LEAD_DAYS}일')
print(f'  - 발령 기준: 2회 연속 임계값 초과')
print(f'  - Train: 2016-01-05 ~ 2021-12-31')
print(f'  - Val  : 2022-01-01 ~ 2023-12-31')
print(f'  - Test : 2024-01-01 ~ 2025-12-22')
print()
print('▶ 주요 피처')
print(f'  기상: 기온·일사량·일조시간·강수량·습도·풍속·전운량')
print(f'  수질: 수온·pH·DO·Chl-a·탁도·투명도')
print(f'  수문: 저수위·저수량·저수율·유입량·방류량')
print(f'  엔지니어링: lag(1,3,7,14일)·rolling(7,14,30일)·계절sin/cos·교호작용')
print()
print('▶ 예측 성능 (테스트 기간, LightGBM + 후처리)')
acc = accuracy_score(y_test, y_pred_combined)
mf1 = f1_score(y_test, y_pred_combined, average='macro', zero_division=0)
wf1 = f1_score(y_test, y_pred_combined, average='weighted', zero_division=0)
경계r = recall_score(y_test, y_pred_combined, labels=[2], average='macro', zero_division=0)
print(f'  Accuracy:        {acc:.4f}')
print(f'  Macro F1-Score:  {mf1:.4f}')
print(f'  Weighted F1:     {wf1:.4f}')
print(f'  경계 Recall:     {경계r:.4f}')
print()
print('▶ 경보 적중률')
actual_a = (y_test.values >= 1).sum()
hit_a    = ((y_test.values >= 1) & (y_pred_combined >= 1)).sum()
print(f'  Hit Rate: {hit_a}/{actual_a} ({hit_a/max(actual_a,1)*100:.1f}%)')
print()
print('▶ 주요 영향 인자 (SHAP)')
for i, row in shap_importance.head(7).iterrows():
    print(f'  {i+1}. {row["feature"]}: {row["shap_importance"]:.5f}')
print()
print('▶ 데이터 출처')
print('  1. 한국수자원공사(K-water): 대청댐 주간 조류모니터링·수문운영정보')
print('  2. 기상청 기상자료개방포털: 대전·청주·보은 관측소 일별 기상자료')
print('  3. 기후에너지환경부: 조류경보제 운영지침 (발령기준 적용)')
print('=' * 70)

In [ ]:
import pickle

artifacts = {
    'lgb_model':       lgb_model,
    'xgb_model':       xgb_model,
    'reg_model':       reg_model,
    'feature_cols':    feature_cols,
    'alert2idx':       ALERT2IDX,
    'idx2alert':       IDX2ALERT,
    'alert_order':     ALERT_ORDER,
    'thresholds':      THRESHOLDS,
    'lead_days':       LEAD_DAYS,
    'consec_days':     CONSEC_DAYS,
    'best_params':     best_params,
    'shap_importance': shap_importance,
    'n_classes':       N_CLASSES,
}
with open('algae_model_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

# 테스트 예측 결과 저장
tout = df_ms[te_mask][['조사일', '채수위치']].copy().reset_index(drop=True)
tout['실제_발령단계']  = y_test.reset_index(drop=True).map(IDX2ALERT)
tout['예측_발령단계']  = pd.Series(y_pred_combined).map(IDX2ALERT)
tout['예측_세포수']    = y_pred_cyano.round(1)
tout['미발령_확률']    = y_prob_lgb[:, 0].round(4)
tout['관심_확률']      = y_prob_lgb[:, 1].round(4)
tout['경계_확률']      = y_prob_lgb[:, 2].round(4)
tout['대발생_확률']    = y_prob_lgb[:, 3].round(4)
tout.to_csv('test_predictions.csv', index=False, encoding='utf-8-sig')

print('저장 완료 (algae_model_artifacts.pkl, test_predictions.csv, *.png)')